# Mars image-only replacement audit

## Goal
Verify the image-only change from `orbital_300_v3` to `orbital_300_v4` and the complete `crater_annotations_v5` candidate set. No network, models, training, or annotation mutation. This is a technical provenance audit, not scientific model results.

## Setup
Run from the project root (or its `notebooks/` directory), with the benchmark CPU dependencies. Stored outputs were executed top-to-bottom as ordinary Python cells in the local benchmark interpreter, not a Jupyter/Colab kernel. Runtime/GPU Colab validation remains pending.

In [1]:
from pathlib import Path
import json, sys, platform
ROOT = Path.cwd().resolve()
if not (ROOT / 'configs/crater_annotation_v2.toml').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'configs/crater_annotation_v2.toml').is_file(), 'Run from project root or notebooks/'
sys.path.insert(0, str(ROOT / 'src'))
from planetary_vlm.io import read_jsonl
from planetary_vlm.datasets.orbital_questions import file_hash, digest
from planetary_vlm.datasets.image_replacement import verify_replacement
from planetary_vlm.datasets.crater_annotations import verify_crater_build, load_crater_requests
print({'python':platform.python_version(), 'root':str(ROOT), 'execution':'ordinary Python cells; no GPU'})


{'python': '3.12.14', 'root': 'C:\\Users\\Timur\\myprojects\\robo-projects\\article', 'execution': 'ordinary Python cells; no GPU'}


## Steps
The replacement is an image-feature-registered crop from a different published THEMIS mosaic version, not inpainting and not a clean/corrupt counterfactual from one acquisition. Original CVAT geometry stays anchored to the old image; the new image is registered to that pixel grid. Source-only visibility inspection resolves the former missing region without accepting all scoring targets.

In [2]:
old = ROOT / 'data_orbital/orbital_300_v3'
new = ROOT / 'data_orbital/orbital_300_v4'
row = next(r for r in read_jsonl(new / 'mars/segmentation_manifest.jsonl') if r['sample_id']=='mars_seg_018')
proof = verify_replacement(ROOT, row)
assert proof['mask_sha256'] == row['mask_sha256']
print({'sample_id':row['sample_id'], 'image_sha256':row['image_sha256'], 'record_sha256':digest(proof),
       'output_size':proof['registration']['output_size'], 'median_residual_px':proof['registration']['median_residual_px'],
       'p95_residual_px':proof['registration']['p95_residual_px'], 'artificial_border_fill':proof['registration']['artificial_border_fill_used']})


{'sample_id': 'mars_seg_018', 'image_sha256': '6cc55c41772dab16331364bcda9f12b90af02338dc9a89d2f015523bbe86889f', 'record_sha256': '3488edce1f225bf7be55905bce09d0610428dcb2443df402dd98f2ecfa618f4a', 'output_size': [512, 512], 'median_residual_px': 0.25071610788851184, 'p95_residual_px': 0.7651964244334705, 'artificial_border_fill': False}


In [3]:
manifests = {f'{p}/{t}_manifest.jsonl' for p in ('moon','mars') for t in ('segmentation','height','train')}
assets = [p.relative_to(old) for p in old.rglob('*') if p.is_file() and p.relative_to(old).as_posix() not in manifests | {'audit.json'}]
changed = [p.as_posix() for p in assets if file_hash(old/p) != file_hash(new/p)]
assert len(assets)==800 and changed==['mars/segmentation/018/image.png']
protected_assets = [p for p in assets if 'mask' in p.name or p.suffix=='.npy']
assert all(file_hash(old/p)==file_hash(new/p) for p in protected_assets)
print({'compared_data_assets':len(assets),'changed':changed,'unchanged':len(assets)-len(changed),
       'all_masks_and_heights_identical':True,'protected_asset_count':len(protected_assets)})


{'compared_data_assets': 800, 'changed': ['mars/segmentation/018/image.png'], 'unchanged': 799, 'all_masks_and_heights_identical': True, 'protected_asset_count': 500}


In [4]:
folder = ROOT / 'data_orbital/crater_annotations_v5'
verified = verify_crater_build(ROOT, folder)
inputs, metadata = load_crater_requests(ROOT, folder)
a = {q['request_id']:q for q in read_jsonl(ROOT / 'data_orbital/crater_annotations_v4/questions.jsonl')}
b = read_jsonl(folder / 'questions.jsonl')
changed_answers = [{'request_id':q['request_id'],'old':a[q['request_id']]['candidate_answer'],'new':q['candidate_answer']} for q in b if a[q['request_id']]['candidate_answer']!=q['candidate_answer']]
assert len(inputs)==len(b)==400 and metadata['images']==100
assert metadata['invariant_candidate_answers']==400 and metadata['unknown_references']==0
assert metadata['reference_ready_questions']==0
assert len(changed_answers)==3 and all(r['request_id'].startswith('mars_seg_018__') and r['old'] is None for r in changed_answers)
assert all(not r.modality_paths and len(r.image_paths)==1 for r in inputs)
print({'images':metadata['images'],'candidates':400,'unknown':0,'accepted_scoring_targets':0,
       'build_sha256':verified['bundle_sha256'],'changed_answers':changed_answers})


{'images': 100, 'candidates': 400, 'unknown': 0, 'accepted_scoring_targets': 0, 'build_sha256': '6f4a5432e3f30fa0a8cd8e243952ef1e53e4e03f001552248c40b870025f676e', 'changed_answers': [{'request_id': 'mars_seg_018__crater_center_v1__clean', 'old': None, 'new': 'YES'}, {'request_id': 'mars_seg_018__crater_largest_location_v1__clean', 'old': None, 'new': '5'}, {'request_id': 'mars_seg_018__crater_half_count_v1__clean', 'old': None, 'new': 'LEFT'}]}


## Checks
All 799 other data assets are byte-identical, including every mask/height asset. Only the three formerly unresolved Mars018 answers were added; the other 397 candidates did not change. No accepted GT was manufactured to reach 400.

## Next Steps
Use the GT-free ZIP with `spacellava_q1_q4_colab.ipynb`. Finalize references before inspecting model predictions. Actual Colab install, GPU smoke, Drive checkpoints, and full inference require an interactive Colab session. Image-source license review and model-training overlap audit remain separate open items.